In [1]:
import os
import numpy as np
import pandas as pd

EMBEDDING_DIR = "/home/pc/LSC24_SemanticSearchWebApp/backend/data/embeddings/keyframe_clips"
COLLECTION_NAME = "vbs25_clips"
ID_MAPPING = "/home/pc/LSC24_SemanticSearchWebApp/backend/data/id_mapping/keyframes.csv"

In [9]:
from pymilvus import connections, utility, MilvusException, Collection
connections.connect(host="localhost", port="19530")
try:
    collections = utility.list_collections()
    print("List of collections: ", collections)
except MilvusException as e:
    print(e)


from pymilvus import MilvusClient, DataType
CLUSTER_ENDPOINT = "http://localhost:19530"
TOKEN = "root:Milvus"
client = MilvusClient(uri=CLUSTER_ENDPOINT, token=TOKEN)

List of collections:  ['vbs25_bgem3', 'vbs25_clips']


In [10]:
collection = Collection('vbs25_clips')
collection.num_entities
client.get_collection_stats(collection_name="vbs25_clips")

{'row_count': 0}

In [29]:
# if not os.path.exists(ID_MAPPING):
#     df = pd.DataFrame(columns=["keyframe_name", "keyframe_id"])
#     df.to_csv(ID_MAPPING, index=False)
#     print(f"Created {ID_MAPPING}")

Created /home/pc/LSC24_SemanticSearchWebApp/backend/data/id_mapping/keyframes.csv


In [30]:
# id_mapping = pd.read_csv(ID_MAPPING)
# id_mapping.set_index("keyframe_name", inplace=True)
# CURR_ID = len(id_mapping)
# print("Current ID: ", CURR_ID)

Current ID:  0


In [16]:
import math

def get_keyframe_order_from_name(name):
    return int(name.split('_')[1])

def get_keyframe_name_from_entry_name(name):
    keyframe_order = int(name.split("_")[1])
    context_order = math.ceil(keyframe_order / 16)
    video_order = int(name[4:9])
    return f"V3C/{video_order:05d}/{context_order:05d}/{keyframe_order:05d}"

id_mapping = pd.read_csv(ID_MAPPING, index_col=0)

id_mapping_updates = []

for entry in sorted(os.scandir(EMBEDDING_DIR), key=lambda e: e.name)[2:3]:
    if entry.is_dir():
        clip_name = entry.name
        clip_dir = os.path.join(EMBEDDING_DIR, clip_name)
        for clip_entry in sorted(os.scandir(clip_dir), key=lambda e: get_keyframe_order_from_name(e.name)):
            if clip_entry.is_file() and clip_entry.name.endswith('.npy'):
                name = clip_entry.name
                path = os.path.join(clip_dir, name)
                keyframe_name = get_keyframe_name_from_entry_name(name)
                embedding = np.load(path).astype(np.float32)

                # Insert data to Milvus
                data = [{
                    "keyframe_id": id_mapping.loc[keyframe_name, "keyframe_id"],
                    "keyframe_name": keyframe_name,
                    "embedding": embedding,
                }]
                # collection.insert(data=data)
                res = client.insert(collection_name=COLLECTION_NAME, data=data)
                print(res)
                print(data[0]["keyframe_name"], data[0]["keyframe_id"])
            else:
                print("Not a file")

{'insert_count': 1, 'ids': [190], 'cost': 0}
V3C/00003/00001/00001 190
{'insert_count': 1, 'ids': [191], 'cost': 0}
V3C/00003/00001/00002 191
{'insert_count': 1, 'ids': [192], 'cost': 0}
V3C/00003/00001/00003 192
{'insert_count': 1, 'ids': [193], 'cost': 0}
V3C/00003/00001/00004 193
{'insert_count': 1, 'ids': [194], 'cost': 0}
V3C/00003/00001/00005 194
{'insert_count': 1, 'ids': [195], 'cost': 0}
V3C/00003/00001/00006 195
{'insert_count': 1, 'ids': [196], 'cost': 0}
V3C/00003/00001/00007 196
{'insert_count': 1, 'ids': [197], 'cost': 0}
V3C/00003/00001/00008 197
{'insert_count': 1, 'ids': [198], 'cost': 0}
V3C/00003/00001/00009 198
{'insert_count': 1, 'ids': [199], 'cost': 0}
V3C/00003/00001/00010 199
{'insert_count': 1, 'ids': [200], 'cost': 0}
V3C/00003/00001/00011 200
{'insert_count': 1, 'ids': [201], 'cost': 0}
V3C/00003/00001/00012 201
{'insert_count': 1, 'ids': [202], 'cost': 0}
V3C/00003/00001/00013 202
{'insert_count': 1, 'ids': [203], 'cost': 0}
V3C/00003/00001/00014 203
{'inse

In [19]:
# client.get(collection_name="vbs25_clips", ids=[1, 2, 3])[0]['embedding'].__len__()
client.get(collection_name="vbs25_clips", ids=[190])

data: ["{'embedding': [0.018742798, -0.04722941, 0.015800118, 0.017350825, 0.05157627, -0.0015148394, -0.022735564, -0.02427406, -0.017485138, 0.026130024, -0.053383395, -0.001875806, -0.07995299, 0.0034463548, -0.00963392, 0.005503789, 0.033969034, 0.08498363, -0.07565497, -0.010091806, 0.03431092, -0.00062234333, 0.0041759196, 0.007869533, 0.010903791, -0.03877989, 0.008077108, 0.030281523, -0.019768462, -0.40166977, -0.006160092, -0.05226005, 0.021551166, 0.0012851332, -0.026349809, 0.0047833817, -0.015165183, 0.029036073, -0.011013683, 0.01714325, -0.010110121, -0.007234598, -0.011929455, 0.011593672, 0.061149143, 0.0045758067, 0.01337027, 0.021917474, -0.0021078016, -0.012021032, -0.022882087, -0.017814815, -0.03018384, 0.025568351, 0.0016697574, -0.0066301883, -0.034139976, -0.0107572675, 0.0042003402, 0.00752154, -0.029524485, -0.007698589, 0.004032449, -0.024835734, 0.0010966369, -0.049060952, 0.021441272, -0.006404298, -0.007289544, 0.030843196, 0.025739295, 0.012845227, 0.042